# Train VeraVoice on Kaggle

Runs the whole retrain on a free Kaggle GPU in about an hour, most of it WavLM feature extraction.

**Before you press Run all:**
1. *Settings → Accelerator → GPU T4 x2* (one GPU is used) and *Settings → Internet → On*.
2. *Add Input* → search **ASVspoof 2019** and add a copy that contains the `LA` folder with `ASVspoof2019_LA_cm_protocols` inside.
3. *Add Input* → search **In-the-Wild audio deepfake** and add a copy that contains `meta.csv` (optional, but it's the number that matters most).

The scripts search `/kaggle/input` for the files they need, so the dataset names and nesting don't matter.

**When it finishes**, download `veravoice_model.zip` from the Output panel. It holds `model/head.pt`, `model/config.json` and `model/report.json`.

In [ ]:
BRANCH = "retrain-wavlm"  # switch to "main" once the retrain PR is merged
!git clone -q -b {BRANCH} https://github.com/OjalSingh/VeraVoice_Cyberverse.git veravoice
%cd veravoice
!pip install -q "transformers>=4.40" librosa soundfile
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be slow)")

## Optional: modern TTS from MLAAD

ASVspoof 2019 only has 2019-era attacks. MLAAD adds clips from dozens of newer TTS systems (XTTS, VITS, Bark and more). We take English only, so the model can't learn "this language means fake", and hold out 30% of the TTS systems as an unseen-attack test set. Set `USE_MLAAD = False` to skip it.

In [ ]:
USE_MLAAD = True
MLAAD_PATTERNS = ["fake/en/**"]   # adjust if the dataset layout changes
MLAAD_DIR = "/kaggle/working/mlaad"
if USE_MLAAD:
    try:
        from huggingface_hub import snapshot_download
        snapshot_download("mueller91/MLAAD", repo_type="dataset", local_dir=MLAAD_DIR, allow_patterns=MLAAD_PATTERNS)
    except Exception as e:
        print("MLAAD download failed, continuing without it:", e)
        USE_MLAAD = False

In [ ]:
mlaad_arg = f"--mlaad {MLAAD_DIR}" if USE_MLAAD else ""
!python scripts/build_manifests.py --asvspoof /kaggle/input --itw /kaggle/input {mlaad_arg} --max-per-source 30000 --out /kaggle/working/manifests

## Extract WavLM features (the slow part)

In [ ]:
import os
M, F = "/kaggle/working/manifests", "/kaggle/working/feats"
for split in ["train", "dev", "test_asvspoof", "test_mlaad", "test_itw"]:
    if os.path.exists(f"{M}/{split}.json"):
        crop = "--random-crop" if split == "train" else ""
        !python scripts/extract_embeddings.py {M}/{split}.json {F}/{split}.npz --batch-size 64 {crop}

## Train the head and evaluate

In [ ]:
!python scripts/train_head.py {F}/train.npz {F}/dev.npz --out /kaggle/working/model
tests = " ".join(f"{F}/{s}.npz" for s in ["dev", "test_asvspoof", "test_mlaad", "test_itw"] if os.path.exists(f"{F}/{s}.npz"))
!python scripts/evaluate.py /kaggle/working/model {tests} --report /kaggle/working/model/report.json

In [ ]:
!cd /kaggle/working && zip -qr veravoice_model.zip model && ls -la model veravoice_model.zip